# 유물 시스템 파라미터 탐색 v2 — 3성 달성 가능 버전

## 목표
**'열심히 하면 3성 달성 가능'** 설계 기준으로 파라미터 탐색

### v1과의 차이점
| 항목 | v1 (3억 탐색) | v2 (1억 탐색) |
|------|-------------|-------------|
| n_3_100 목표 | 0명 (사실상 불가) | **1~3명 필수** |
| 3성 비용 배율 | 최대 ×4.0 | **최대 ×2.0** |
| 2→3성 성급 비용 | 최대 12,500P | **최대 3,000P** |
| 피트니스 | 분포 피팅 | **3성 도달 필수 + 분포 균형** |

## GPU 켜는 법
```
런타임 → 런타임 유형 변경 → 하드웨어 가속기 → GPU → 저장
```

## 사용 순서
1. GPU 런타임 설정
2. 전체 셀 순서대로 실행
3. Cell 2에서 랭킹 TXT 파일 업로드
4. 결과 CSV 자동 다운로드

In [ ]:
# ============================================================
# 0. 환경/GPU 확인
# ============================================================
import sys, os, re, csv, math, heapq, time, json
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Tuple, Optional

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
        get_ipython().system('nvidia-smi')
except Exception as e:
    print('torch 확인 오류:', e)

USE_GPU = False
try:
    import cupy as cp
    _ = cp.arange(3).sum()
    USE_GPU = True
    xp = cp
    print('CuPy 사용 가능: GPU batch 연산 활성화')
except Exception as e:
    xp = np
    print('CuPy 사용 불가 → NumPy CPU fallback:', repr(e))

print('준비 완료')

In [ ]:
# ============================================================
# 1. 랭킹 TXT 파일 업로드 (rank.chosung.app Production DB)
# ============================================================
# 업로드할 파일: 랭킹_및_파라미터.txt
# (Replit에서 다운로드한 파일)

try:
    from google.colab import files
    print('Colab 환경: TXT 파일을 업로드하세요.')
    uploaded = files.upload()
    TXT_PATH = list(uploaded.keys())[0]
    print(f'업로드 완료: {TXT_PATH}')
except Exception:
    # 로컬 환경 fallback
    candidates = list(Path('.').glob('*.txt'))
    if candidates:
        TXT_PATH = str(candidates[0])
        print(f'로컬 파일 사용: {TXT_PATH}')
    else:
        TXT_PATH = None
        print('TXT 파일을 찾을 수 없습니다.')

In [ ]:
# ============================================================
# 2. 랭킹 데이터 파싱
# ============================================================
# TXT 파일에서 "7. 랭킹 데이터" 섹션 이후의
# 순위/닉네임/점수 컬럼을 파싱합니다.

def parse_ranking_txt(path: str) -> np.ndarray:
    """TXT 파일에서 점수 배열 파싱 (100P 이상 유저만)"""
    scores = []
    in_data = False
    with open(path, encoding='utf-8') as f:
        for line in f:
            line = line.rstrip()
            # 데이터 섹션 시작 감지
            if '순위' in line and '닉네임' in line and '점수' in line:
                in_data = True
                continue
            if not in_data:
                continue
            # 구분선 스킵
            if set(line.strip()) <= set('─ '):
                continue
            # END 감지
            if 'END' in line or '════' in line:
                break
            # 숫자로 시작하는 행 파싱
            parts = line.split()
            if not parts:
                continue
            try:
                rank = int(parts[0])
                # 점수는 P로 끝나는 토큰 or 3번째 숫자 토큰
                for tok in parts[2:]:
                    tok_clean = tok.replace(',', '').replace('P', '')
                    if tok_clean.isdigit():
                        score = int(tok_clean)
                        if score >= 100:
                            scores.append(score)
                        break
            except (ValueError, IndexError):
                continue
    arr = np.array(sorted(scores, reverse=True), dtype=np.float64)
    print(f'파싱 완료: {len(arr)}명 (100P 이상)')
    print(f'최고 {arr[0]:,.0f}P / 중앙값 {np.median(arr):,.0f}P / 최저 {arr[-1]:,.0f}P')
    return arr

if TXT_PATH:
    SCORES = parse_ranking_txt(TXT_PATH)
else:
    # fallback: 실제 Production 데이터 상위 샘플 하드코딩
    print('TXT 없음 → 하드코딩 상위 데이터 사용')
    SCORES = np.array([
        160060,60395,48891,35768,26313,25862,24263,21607,16499,16399,
        15600,15127,14974,14904,13638,10832,10500,10092,9480,9441,
        9059,9039,8643,8347,7516,7475,7427,7005,6657,6641,
        6610,6593,6316,6296,6288,6281,6276,6276,6263,6190,
        5919,5887,5879,5846,5318,5236,5233,5138,5084,5074,
        5073,5073,5072,5001,4924,4896,4856,4853,4806,4758,
        4714,4658,4639,4610,4583,4476,
        3000,2800,2600,2400,2200,2000,1800,1600,1400,1200,
        1000,900,800,700,600,500,400,300,200,150,130,110,100
    ], dtype=np.float64)
    print(f'하드코딩: {len(SCORES)}명')

N_USERS = len(SCORES)
print(f'총 {N_USERS}명 데이터 로드')

In [ ]:
# ============================================================
# 3. 조건부 마르코프 체인 기댓값 계산 함수
# ============================================================
#
# T[n] = (c(n) + p_fail * T[n-1]) / p_success
# T[0] = c(0) / p_success  (실패해도 0강 유지)
# E[0→N강] = sum(T[n], n=0..N-1)
#
# 성급강화 E[promo] = (cost + p_fail * E[복귀후→max]) / p_success
# E[복귀후→max] = 가중평균(실패레벨 분포 × 잔여강화 기댓값)

# ── 현재 파라미터 기준값 ─────────────────────────────────────

# 강화 확률 프리셋 (success%, keep%, fail%)
PROB_PRESETS = {
    # 1성
    '1_easy':      [(0,5,92,7,1),(6,15,87,11,2),(16,25,81,14,5),(26,35,75,18,7),(36,45,70,21,9),(46,50,65,23,12)],
    '1_current':   [(0,5,87,11,2),(6,15,82,13,5),(16,25,76,16,8),(26,35,70,19,11),(36,45,65,21,14),(46,50,60,23,17)],
    '1_mild_hard': [(0,5,83,13,4),(6,15,78,15,7),(16,25,72,18,10),(26,35,66,21,13),(36,45,61,23,16),(46,50,56,25,19)],
    '1_hard':      [(0,5,78,15,7),(6,15,73,17,10),(16,25,67,20,13),(26,35,61,23,16),(36,45,56,25,19),(46,50,51,27,22)],
    # 2성
    '2_easy':      [(0,20,88,10,2),(21,40,81,13,6),(41,60,74,18,8),(61,70,67,22,11),(71,80,61,25,14)],
    '2_current':   [(0,20,84,13,3),(21,40,77,16,7),(41,60,70,20,10),(61,70,63,24,13),(71,80,57,27,16)],
    '2_mild_hard': [(0,20,80,15,5),(21,40,73,18,9),(41,60,66,22,12),(61,70,59,26,15),(71,80,53,29,18)],
    '2_hard':      [(0,20,75,18,7),(21,40,68,21,11),(41,60,61,25,14),(61,70,54,28,18),(71,80,48,31,21)],
    # 3성
    '3_easy':      [(0,20,86,11,3),(21,40,79,14,7),(41,60,72,17,11),(61,80,66,21,13),(81,90,61,26,13),(91,100,57,30,13)],
    '3_current':   [(0,20,82,13,5),(21,40,75,16,9),(41,60,68,19,13),(61,80,62,23,15),(81,90,57,28,15),(91,100,53,32,15)],
    '3_mild_hard': [(0,20,78,15,7),(21,40,71,18,11),(41,60,64,21,15),(61,80,58,25,17),(81,90,53,30,17),(91,100,49,34,17)],
    '3_hard':      [(0,20,73,18,9),(21,40,66,21,13),(41,60,59,24,17),(61,80,53,28,19),(81,90,48,33,19),(91,100,44,37,19)],
}

def get_prob(preset_key: str, star: int, n: int) -> Tuple[float, float]:
    """(success_rate, fail_rate) 반환 (0~1 소수)"""
    key = f'{star}_{preset_key}'
    if key not in PROB_PRESETS:
        key = f'{star}_current'
    for (lo, hi, s, k, f) in PROB_PRESETS[key]:
        if lo <= n <= hi:
            tot = s + k + f
            return s/tot, f/tot
    return 0.6, 0.17

# 강화 비용 계산
def calc_cost(star: int, n: int, mult: float) -> float:
    if star == 1:
        if n <= 10:  base = 2*n
        elif n <= 20: base = 20 + 2*(n-10)
        elif n <= 30: base = 40 + 3*(n-20)
        elif n <= 40: base = 70 + 5*(n-30)
        else:         base = 120 + 7*(n-40)
    elif star == 2:
        if n <= 40:  base = 15 + 3*n
        elif n <= 60: base = 135 + 6*(n-40)
        elif n <= 70: base = 255 + 10*(n-60)
        else:         base = 355 + 15*(n-70)
    else:  # star == 3
        if n <= 20:  base = 40 + 3*n
        elif n <= 40: base = 100 + 5*(n-20)
        elif n <= 60: base = 200 + 7*(n-40)
        elif n <= 80: base = 340 + 11*(n-60)
        elif n <= 90: base = 560 + 16*(n-80)
        else:         base = 720 + 22*(n-90)
    return round(base * mult)

# 성급강화 실패 복귀 레벨 분포
# 1성 실패 → 30~39강, 2성 실패 → 50~59강 (동일 분포)
FAIL_PROBS  = np.array([18,16,14,12,10,9,7,6,5,3], dtype=np.float64)
FAIL_PROBS /= FAIL_PROBS.sum()
STAR1_FAIL_LEVELS = list(range(30, 40))  # 30~39강
STAR2_FAIL_LEVELS = list(range(50, 60))  # 50~59강

# 마르코프 체인 T[n] 계산
def compute_T(star: int, max_lv: int, prob_preset: str, cost_mult: float) -> np.ndarray:
    T = np.zeros(max_lv)
    for n in range(max_lv):
        s, f = get_prob(prob_preset, star, n)
        c = calc_cost(star, n, cost_mult)
        prev = T[n-1] if n > 0 else 0.0
        T[n] = (c + f * prev) / s
    return T

def compute_promo_E(promo_cost: float, p_succ: float,
                    T: np.ndarray, max_lv: int,
                    fail_levels: list) -> float:
    """성급강화 기댓 비용"""
    fail_cost = 0.0
    for i, k in enumerate(fail_levels):
        regain = T[k:max_lv].sum()
        fail_cost += FAIL_PROBS[i] * regain
    return (promo_cost + (1 - p_succ) * fail_cost) / p_succ

def compute_all_E(p1: str, p2: str, p3: str,
                  c1: float, c2: float, c3: float,
                  promo12_cost: float, promo12_rate: float,
                  promo23_cost: float, promo23_rate: float) -> dict:
    """전체 기댓값 계산"""
    T1 = compute_T(1, 50, p1, c1)
    T2 = compute_T(2, 80, p2, c2)
    T3 = compute_T(3, 100, p3, c3)

    e1   = T1.sum()                                                       # 1성 0→50강
    pe12 = compute_promo_E(promo12_cost, promo12_rate, T1, 50, STAR1_FAIL_LEVELS)  # 성급 1→2
    e2   = T2.sum()                                                       # 2성 0→80강
    pe23 = compute_promo_E(promo23_cost, promo23_rate, T2, 80, STAR2_FAIL_LEVELS)  # 성급 2→3
    e3   = T3.sum()                                                       # 3성 0→100강

    ex_1_50   = e1
    ex_2_entry = e1 + pe12          # 2성 진입(성급1→2 포함) 누적
    ex_2_80   = ex_2_entry + e2    # 2성 80강 완성 누적
    ex_3_entry = ex_2_80 + pe23    # 3성 진입(성급2→3 포함) 누적
    ex_3_100  = ex_3_entry + e3    # 3성 100강 완성 누적

    return {
        'ex_1_50':    ex_1_50,
        'ex_2_entry': ex_2_entry,
        'ex_2_80':    ex_2_80,
        'ex_3_entry': ex_3_entry,
        'ex_3_100':   ex_3_100,
    }

# 테스트
test = compute_all_E('current','current','current', 1.0,1.0,1.0, 190,0.55, 505,0.55)
print('현재 파라미터 기준 E[X]:')
for k,v in test.items():
    print(f'  {k}: {v:,.0f}P')

In [ ]:
# ============================================================
# 4. 보너스 프로필 (유물 강화 보너스율 → 추가 포인트 발행)
# ============================================================

def calc_bonus_rate(star: int, level: int) -> float:
    if star == 1: return min(25.0,  5.0  + 0.4  * level)
    if star == 2: return min(50.0,  15.0 + 0.45 * level)
    return              min(80.0,  30.0 + 0.5  * level)

BONUS_PROFILES = {
    'low':  {'max_bonus': 15.0, 'note': '1성 상한 축소'},
    'mid':  {'max_bonus': 25.0, 'note': '현재 기준'},
    'high': {'max_bonus': 35.0, 'note': '보너스 상향'},
}

def compute_bonus_stats(scores: np.ndarray, ex_1_50: float, ex_2_80: float,
                        ex_3_100: float, bonus_profile: str) -> dict:
    """유저별 추정 강화 단계에 따른 보너스율 분포 계산"""
    # 점수 기반 단계 추정 (점수 ≈ 총 획득 포인트)
    bonus_rates = []
    for sc in scores:
        if sc >= ex_3_100:
            star, lv = 3, 100
        elif sc >= ex_2_80:
            star, lv = 3, min(100, int((sc - ex_2_80) / max(1, ex_3_100 - ex_2_80) * 100))
        elif sc >= ex_2_80 * 0.5:
            star, lv = 2, min(80, int((sc / max(1, ex_2_80)) * 80))
        elif sc >= ex_1_50:
            star, lv = 2, min(30, int((sc - ex_1_50) / max(1, ex_2_80 - ex_1_50) * 80))
        elif sc >= ex_1_50 * 0.5:
            star, lv = 1, min(50, int((sc / max(1, ex_1_50)) * 50))
        else:
            star, lv = 1, min(30, int(sc / max(1, ex_1_50) * 50))
        bonus_rates.append(calc_bonus_rate(star, lv))

    rates = np.array(bonus_rates)
    # 추가 발행률 = 평균 보너스율 (퀴즈 포인트에서 추가로 발행되는 비율)
    mean_rate = rates.mean()
    median_rate = np.median(rates)
    p95_rate = np.percentile(rates, 95)
    max_rate = rates.max()

    return {
        'mean_bonus_pct':   mean_rate,
        'median_bonus_pct': median_rate,
        'p95_bonus_pct':    p95_rate,
        'max_bonus_pct':    max_rate,
        'inflation_pct':    mean_rate,  # 추가 발행률
    }

print('보너스 프로필 정의 완료')

In [ ]:
# ============================================================
# 5. 피트니스 함수 (v2: 3성 달성 가능 버전)
# ============================================================
#
# v1과의 핵심 차이:
#   v1: 분포 피팅 최소화 (3성이 0명이어도 OK)
#   v2: n_3_100 >= 1명 필수 + 분포 균형 + 인플레이션 제어
#
# 목표 인원:
#   n_1_50:   30~80명  (전체의 1.8~4.8%)
#   n_2_80:   5~20명   (전체의 0.3~1.2%)
#   n_3_100:  1~5명    (전체의 0.06~0.3%) ← 필수 조건

# 도달 가능 인원 계산
def count_reachable(scores: np.ndarray, threshold: float) -> int:
    return int((scores >= threshold).sum())

# v2 피트니스 함수 (낮을수록 좋음)
def fitness_v2(E: dict, scores: np.ndarray, bonus_stats: dict) -> float:
    n1 = count_reachable(scores, E['ex_1_50'])
    n2 = count_reachable(scores, E['ex_2_80'])
    n3 = count_reachable(scores, E['ex_3_100'])

    # ── 하드 제약: 3성 달성 가능 (1명 이상 필수) ─────────────
    if n3 < 1:
        return 1e9  # 완전 제거

    # ── 목표 인원 오차 (로그 스케일 오차²) ───────────────────
    # 목표: n1=50명, n2=10명, n3=2명
    TARGET_N1, TARGET_N2, TARGET_N3 = 50, 10, 2
    W_N1, W_N2, W_N3 = 1.0, 2.0, 4.0  # 3성에 더 높은 가중치

    def log_err(actual, target):
        if actual <= 0 or target <= 0: return 10.0
        return (math.log(actual / target)) ** 2

    err_n = (W_N1 * log_err(n1, TARGET_N1) +
             W_N2 * log_err(n2, TARGET_N2) +
             W_N3 * log_err(n3, TARGET_N3))

    # ── E[X] 비율 균형 ────────────────────────────────────────
    # 이상적 비율: ex_2_80 / ex_1_50 ≈ 3~6, ex_3_100 / ex_2_80 ≈ 3~8
    ratio_12 = E['ex_2_80'] / max(1, E['ex_1_50'])
    ratio_23 = E['ex_3_100'] / max(1, E['ex_2_80'])

    # 비율이 너무 극단적이면 패널티
    TARGET_R12, TARGET_R23 = 4.0, 5.0
    err_ratio = (math.log(ratio_12 / TARGET_R12))**2 + (math.log(ratio_23 / TARGET_R23))**2

    # ── 인플레이션 제어 ────────────────────────────────────────
    # 평균 보너스율 5~15% 범위 목표
    inf_pct = bonus_stats['inflation_pct']
    TARGET_INF = 8.0
    err_inf = (math.log(max(0.1, inf_pct) / TARGET_INF))**2

    # ── 3성 절대 비용 패널티 ──────────────────────────────────
    # E[3성 완성] > 200,000P이면 추가 패널티 (너무 어려움)
    if E['ex_3_100'] > 200000:
        err_hard = ((E['ex_3_100'] - 200000) / 50000) ** 2
    else:
        err_hard = 0.0

    total = err_n + 0.5 * err_ratio + 0.3 * err_inf + 2.0 * err_hard
    return total

# 테스트
test_E = compute_all_E('current','current','current', 1.0,1.0,1.0, 190,0.55, 505,0.55)
test_bonus = compute_bonus_stats(SCORES, test_E['ex_1_50'], test_E['ex_2_80'], test_E['ex_3_100'], 'mid')
test_score = fitness_v2(test_E, SCORES, test_bonus)
print(f'현재 파라미터 피트니스 점수: {test_score:.4f}')
n1 = count_reachable(SCORES, test_E['ex_1_50'])
n2 = count_reachable(SCORES, test_E['ex_2_80'])
n3 = count_reachable(SCORES, test_E['ex_3_100'])
print(f'  도달 인원: 1성={n1}명, 2성={n2}명, 3성={n3}명')

In [ ]:
# ============================================================
# 6. 파라미터 탐색 공간 정의 (v2: 극단값 제거)
# ============================================================
#
# v1 대비 변경사항:
#   - 3성 비용 배율: 최대 ×2.0  (v1: 최대 ×4.0)
#   - 2→3성 성급 비용: 최대 3,000P  (v1: 12,500P)
#   - 2→3성 성급 성공률: 35%~75%  (v1: 45%)
#   - 보너스 프로필: low/mid/high 모두 탐색

import itertools

# 비용 배율
COST1_MULTS  = [0.5, 0.65, 0.8, 1.0, 1.2, 1.4]      # 1성 비용
COST2_MULTS  = [0.7, 0.85, 1.0, 1.2, 1.4, 1.6]      # 2성 비용
COST3_MULTS  = [0.5, 0.7, 0.9, 1.1, 1.4, 1.7, 2.0]  # 3성 비용 (최대 ×2.0)

# 확률 프리셋
PROB1_PRESETS = ['easy', 'current', 'mild_hard', 'hard']
PROB2_PRESETS = ['easy', 'current', 'mild_hard', 'hard']
PROB3_PRESETS = ['easy', 'current', 'mild_hard', 'hard']

# 성급강화 1→2성
PROMO12_COSTS  = [150, 250, 400, 500, 700, 1000]     # 비용 (P)
PROMO12_RATES  = [0.45, 0.55, 0.65, 0.75]            # 성공률

# 성급강화 2→3성 (v2: 최대 3,000P)
PROMO23_COSTS  = [300, 500, 800, 1200, 1800, 2500, 3000]  # 비용 (P) ← v1 12,500P 제거
PROMO23_RATES  = [0.35, 0.45, 0.55, 0.65, 0.75]           # 성공률

# 보너스 프로필
BONUS_PROFILE_KEYS = ['low', 'mid', 'high']

# 전체 조합 수 계산
total = (len(COST1_MULTS) * len(COST2_MULTS) * len(COST3_MULTS) *
         len(PROB1_PRESETS) * len(PROB2_PRESETS) * len(PROB3_PRESETS) *
         len(PROMO12_COSTS) * len(PROMO12_RATES) *
         len(PROMO23_COSTS) * len(PROMO23_RATES) *
         len(BONUS_PROFILE_KEYS))

print(f'탐색 공간: {total:,}가지 조합')
print(f'  비용배율(1/2/3): {len(COST1_MULTS)}×{len(COST2_MULTS)}×{len(COST3_MULTS)}')
print(f'  확률프리셋(1/2/3): {len(PROB1_PRESETS)}×{len(PROB2_PRESETS)}×{len(PROB3_PRESETS)}')
print(f'  성급(1→2): {len(PROMO12_COSTS)}비용×{len(PROMO12_RATES)}성공률')
print(f'  성급(2→3): {len(PROMO23_COSTS)}비용×{len(PROMO23_RATES)}성공률')
print(f'  보너스프로필: {len(BONUS_PROFILE_KEYS)}')

In [ ]:
# ============================================================
# 7. 탐색 실행 — 랜덤 샘플링 1억 회
# ============================================================
# 전체 조합이 1억을 넘으면 랜덤 샘플링,
# 1억 이하면 완전 탐색

import random

N_SEARCH = 100_000_000  # 1억 회
TOP_K = 200             # 상위 후보 보관
PRINT_EVERY = 5_000_000 # 진행 출력 주기

@dataclass
class Candidate:
    score:       float
    p1:          str
    p2:          str
    p3:          str
    c1:          float
    c2:          float
    c3:          float
    promo12_cost: float
    promo12_rate: float
    promo23_cost: float
    promo23_rate: float
    bonus_profile: str
    ex_1_50:     float
    ex_2_entry:  float
    ex_2_80:     float
    ex_3_entry:  float
    ex_3_100:    float
    n_1_50:      int
    n_2_80:      int
    n_3_100:     int
    mean_bonus:  float
    inflation:   float

# 모든 파라미터 목록
all_p1 = PROB1_PRESETS
all_p2 = PROB2_PRESETS
all_p3 = PROB3_PRESETS
all_c1 = COST1_MULTS
all_c2 = COST2_MULTS
all_c3 = COST3_MULTS
all_pc12 = PROMO12_COSTS
all_pr12 = PROMO12_RATES
all_pc23 = PROMO23_COSTS
all_pr23 = PROMO23_RATES
all_bp = BONUS_PROFILE_KEYS

heap = []  # min-heap (-score, ctr, candidate) — ctr로 동점 비교 충돌 방지
n_valid = 0
_ctr = 0    # 고유 삽입 순서 카운터
t0 = time.time()

random.seed(42)

use_full_search = (total <= N_SEARCH)
print(f'탐색 방식: {"완전 탐색" if use_full_search else f"랜덤 샘플링 {N_SEARCH:,}회"}')

def run_one(p1, p2, p3, c1, c2, c3, pc12, pr12, pc23, pr23, bp):
    try:
        E = compute_all_E(p1, p2, p3, c1, c2, c3, pc12, pr12, pc23, pr23)
        bonus = compute_bonus_stats(SCORES, E['ex_1_50'], E['ex_2_80'], E['ex_3_100'], bp)
        sc = fitness_v2(E, SCORES, bonus)
        return sc, E, bonus
    except Exception:
        return 1e9, {}, {}

if use_full_search:
    iterator = itertools.product(all_p1, all_p2, all_p3,
                                 all_c1, all_c2, all_c3,
                                 all_pc12, all_pr12,
                                 all_pc23, all_pr23, all_bp)
    for i, (p1,p2,p3,c1,c2,c3,pc12,pr12,pc23,pr23,bp) in enumerate(iterator):
        sc, E, bonus = run_one(p1,p2,p3,c1,c2,c3,pc12,pr12,pc23,pr23,bp)
        if sc < 1e8:
            n_valid += 1
            n1 = count_reachable(SCORES, E['ex_1_50'])
            n2 = count_reachable(SCORES, E['ex_2_80'])
            n3 = count_reachable(SCORES, E['ex_3_100'])
            cand = Candidate(sc,p1,p2,p3,c1,c2,c3,pc12,pr12,pc23,pr23,bp,
                             E['ex_1_50'],E['ex_2_entry'],E['ex_2_80'],E['ex_3_entry'],E['ex_3_100'],
                             n1,n2,n3,bonus['mean_bonus_pct'],bonus['inflation_pct'])
            if len(heap) < TOP_K:
                heapq.heappush(heap, (-sc, _ctr, cand))
                _ctr += 1
            elif sc < -heap[0][0]:
                heapq.heapreplace(heap, (-sc, _ctr, cand))
                _ctr += 1
        if (i+1) % PRINT_EVERY == 0:
            elapsed = time.time() - t0
            best = -heap[0][0] if heap else float('inf')
            print(f'[{i+1:,}] 경과 {elapsed:.0f}s | 유효 {n_valid:,} | 현재 최고 {best:.4f}')
else:
    for i in range(N_SEARCH):
        p1  = random.choice(all_p1)
        p2  = random.choice(all_p2)
        p3  = random.choice(all_p3)
        c1  = random.choice(all_c1)
        c2  = random.choice(all_c2)
        c3  = random.choice(all_c3)
        pc12 = random.choice(all_pc12)
        pr12 = random.choice(all_pr12)
        pc23 = random.choice(all_pc23)
        pr23 = random.choice(all_pr23)
        bp  = random.choice(all_bp)

        sc, E, bonus = run_one(p1,p2,p3,c1,c2,c3,pc12,pr12,pc23,pr23,bp)
        if sc < 1e8:
            n_valid += 1
            n1 = count_reachable(SCORES, E['ex_1_50'])
            n2 = count_reachable(SCORES, E['ex_2_80'])
            n3 = count_reachable(SCORES, E['ex_3_100'])
            cand = Candidate(sc,p1,p2,p3,c1,c2,c3,pc12,pr12,pc23,pr23,bp,
                             E['ex_1_50'],E['ex_2_entry'],E['ex_2_80'],E['ex_3_entry'],E['ex_3_100'],
                             n1,n2,n3,bonus['mean_bonus_pct'],bonus['inflation_pct'])
            if len(heap) < TOP_K:
                heapq.heappush(heap, (-sc, _ctr, cand))
                _ctr += 1
            elif sc < -heap[0][0]:
                heapq.heapreplace(heap, (-sc, _ctr, cand))
                _ctr += 1
        if (i+1) % PRINT_EVERY == 0:
            elapsed = time.time() - t0
            best = -heap[0][0] if heap else float('inf')
            print(f'[{i+1:,}/{N_SEARCH:,}] 경과 {elapsed:.0f}s | 유효 {n_valid:,} | 현재 최고 {best:.4f}')

elapsed_total = time.time() - t0
print(f'\n탐색 완료! 총 {elapsed_total:.1f}초 | 유효 후보 {n_valid:,}개')

In [ ]:
# ============================================================
# 8. 상위 후보 정렬 및 출력
# ============================================================

candidates = sorted([c for (_, __, c) in heap], key=lambda x: x.score)
print(f'상위 {len(candidates)}개 후보 (점수 낮을수록 좋음)\n')

best = candidates[0]
print('=== BEST CANDIDATE (v2: 3성 달성 가능) ===')
print(f'score: {best.score:.4f}\n')
print('[파라미터]')
print(f'일반강화 비용 배율: 1성 ×{best.c1}, 2성 ×{best.c2}, 3성 ×{best.c3}')
print(f'확률 프리셋: 1성 {best.p1}, 2성 {best.p2}, 3성 {best.p3}')
print(f'성급강화 1→2: 비용 {best.promo12_cost:.0f}P, 성공률 {best.promo12_rate*100:.1f}%')
print(f'성급강화 2→3: 비용 {best.promo23_cost:.0f}P, 성공률 {best.promo23_rate*100:.1f}%')
print(f'보너스 프로필: {best.bonus_profile}\n')
print('[기대비용]')
print(f'ex_1_50:   {best.ex_1_50:>10,.0f}P  (1성 50강 완성)')
print(f'ex_2_entry:{best.ex_2_entry:>10,.0f}P  (2성 진입 누적)')
print(f'ex_2_80:   {best.ex_2_80:>10,.0f}P  (2성 80강 완성 누적)')
print(f'ex_3_entry:{best.ex_3_entry:>10,.0f}P  (3성 진입 누적)')
print(f'ex_3_100:  {best.ex_3_100:>10,.0f}P  (3성 100강 완성 누적)\n')
print('[도달 가능 인원 (실제 Production 데이터 기준)]')
print(f'n_1_50:  {best.n_1_50}명')
print(f'n_2_80:  {best.n_2_80}명')
print(f'n_3_100: {best.n_3_100}명\n')
print('[경제 지표]')
print(f'평균 보너스율:     {best.mean_bonus:.2f}%')
print(f'포인트 추가발행률: {best.inflation:.2f}%')

print('\n--- 상위 10개 요약 ---')
print(f'{"순위":>4} {"score":>8} {"비용(1/2/3)":>14} {"확률(1/2/3)":>20} {"성급12":>10} {"성급23":>12} {"E[3성]":>10} {"n3":>4}')
for i, c in enumerate(candidates[:10]):
    promo12_str = f'{c.promo12_cost:.0f}P/{c.promo12_rate*100:.0f}%'
    promo23_str = f'{c.promo23_cost:.0f}P/{c.promo23_rate*100:.0f}%'
    cost_str = f'{c.c1}/{c.c2}/{c.c3}'
    prob_str = f'{c.p1[:4]}/{c.p2[:4]}/{c.p3[:4]}'
    print(f'{i+1:>4} {c.score:>8.4f} {cost_str:>14} {prob_str:>20} {promo12_str:>10} {promo23_str:>12} {c.ex_3_100:>10,.0f} {c.n_3_100:>4}')

In [ ]:
# ============================================================
# 9. 결과 CSV 저장 및 다운로드
# ============================================================

rows = [asdict(c) for c in candidates]
df = pd.DataFrame(rows)
df = df.sort_values('score').reset_index(drop=True)
df.index += 1

out_path = 'artifact_params_v2_3성달성가능.csv'
df.to_csv(out_path, index_label='rank', encoding='utf-8-sig')
print(f'CSV 저장 완료: {out_path}')
print(df[['score','c1','c2','c3','p1','p2','p3',
          'promo12_cost','promo12_rate','promo23_cost','promo23_rate',
          'ex_1_50','ex_2_80','ex_3_100','n_1_50','n_2_80','n_3_100']].head(20).to_string())

try:
    from google.colab import files
    files.download(out_path)
    print('다운로드 시작됨')
except Exception:
    print(f'로컬 저장 완료: {out_path}')

In [ ]:
# ============================================================
# 10. 시각화
# ============================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('유물 파라미터 탐색 결과 v2 (3성 달성 가능 버전)', fontsize=14, fontweight='bold')

top20 = candidates[:20]
ranks = list(range(1, len(top20)+1))

# (1) 기댓값 비교
ax = axes[0,0]
ax.bar(ranks, [c.ex_1_50/1000 for c in top20], label='1성 완성', alpha=0.7)
ax.bar(ranks, [(c.ex_2_80 - c.ex_1_50)/1000 for c in top20], bottom=[c.ex_1_50/1000 for c in top20], label='2성 추가', alpha=0.7)
ax.bar(ranks, [(c.ex_3_100 - c.ex_2_80)/1000 for c in top20], bottom=[c.ex_2_80/1000 for c in top20], label='3성 추가', alpha=0.7)
ax.set_xlabel('순위')
ax.set_ylabel('누적 E[X] (천P)')
ax.set_title('단계별 누적 기댓 비용')
ax.legend()

# (2) 도달 인원
ax = axes[0,1]
x = np.arange(len(top20))
ax.plot(ranks, [c.n_1_50 for c in top20], 'o-', label='n_1_50')
ax.plot(ranks, [c.n_2_80 for c in top20], 's-', label='n_2_80')
ax.plot(ranks, [c.n_3_100 for c in top20], '^-', label='n_3_100')
ax.axhline(y=1, color='red', linestyle='--', alpha=0.5, label='3성 최소 1명')
ax.set_xlabel('순위')
ax.set_ylabel('도달 가능 인원')
ax.set_title('단계별 도달 인원')
ax.legend()

# (3) E[3성] 분포
all_e3 = [c.ex_3_100 for c in candidates if c.score < 1e8]
ax = axes[1,0]
ax.hist(all_e3, bins=50, color='steelblue', alpha=0.7)
ax.axvline(x=candidates[0].ex_3_100, color='red', linestyle='--', label=f'Best: {candidates[0].ex_3_100:,.0f}P')
ax.set_xlabel('E[3성 완성] (P)')
ax.set_ylabel('후보 수')
ax.set_title('E[3성 완성] 분포')
ax.legend()

# (4) 피트니스 점수 분포
all_scores = [c.score for c in candidates]
ax = axes[1,1]
ax.plot(range(1, len(all_scores)+1), all_scores, 'o-', markersize=4)
ax.set_xlabel('순위')
ax.set_ylabel('피트니스 점수 (낮을수록 좋음)')
ax.set_title('상위 후보 피트니스 점수')

plt.tight_layout()
plt.savefig('artifact_params_v2_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('차트 저장: artifact_params_v2_chart.png')

try:
    from google.colab import files
    files.download('artifact_params_v2_chart.png')
except Exception:
    pass